In [1]:
import os
import numpy as np
import pandas as pd

In [2]:
def load_dataset(file_path: str) -> pd.DataFrame:
    """Load the raw dataset"""
    if not os.path.exists(file_path):
        raise FileNotFoundError("Dataset not found")
    print(f"loading dataset from: {file_path}")
    return pd.read_csv(file_path)

In [3]:
def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """Data cleaning pipeline"""
    #create a copy to avoid mutating the original dataset
    data = df.copy()
    #drop irrelevent columns
    id_col = ["customerID"]
    data.drop(columns=id_col, inplace=True)
    print(f"dropped column: {id_col}")
    print(df.dtypes)
    print(data.head(5))

    # Convert object columns to lowercase and strip trailing whitespaces
    categorical_cols = data.select_dtypes(include=["str"]).columns
    for col in categorical_cols:
        data[col] = data[col].astype(str).str.strip().str.lower()
    
    # strip duplicates
    duplicate_count = data.duplicated().sum()
    if duplicate_count > 0:
        data.drop_duplicates(inplace=True)
    
    # categorical imputation
    categorical_cols = data.select_dtypes(include=["str"]).columns
    for col in categorical_cols:
        if data[col].isnull().sum() > 0:
            mode_val = data[col].mode()[0]
            data[col] = data[col].fillna(mode_val)
    
    #numerical imputation
    num_cols = data.select_dtypes(include=[np.number]).columns
    for col in num_cols:
        if data[col].isnull().sum() > 0:
            median_val = data[col].median()
            data[col] = data[col].fillna(median_val)
    return data

In [4]:
def save_clean_data(df: pd.DataFrame, output_path: str):
    """Saves the processed Dataframe"""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    df.to_csv(output_path, index=False)
    print(f"Successfully saved to: {output_path}\n")

In [5]:
CLEAN_DATA_PATH = "../data/cleaned_customer_churn.csv"
data = load_dataset("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
clean_df = clean_data(data)
save_clean_data(clean_df, CLEAN_DATA_PATH)

loading dataset from: ../data/WA_Fn-UseC_-Telco-Customer-Churn.csv
dropped column: ['customerID']
customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object
   gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  Female              0     Yes         No       1           No   
1    Male              0      No         No      34          Yes   
2    Male              0      No         No       2          Yes   
3    Male              0    